# 09. 임베딩 사전 계산 — FastAPI 서빙용

실험 1(patron_03, MinMaxScaler) 모델로 전체 47,923개 패턴의 임베딩을 재생성하고 FAISS L2 인덱스를 빌드한다. 이 두 파일을 FastAPI 서버에 배포하면 매 요청마다 임베딩을 재생성하는 3분 대기 없이 즉시 검색이 가능해진다.

## 생성 파일

| 파일 | 크기 | 내용 |
|------|------|------|
| `embeddings.npy` | ~94 MB | (47,923 × 512) float32 임베딩 행렬 |
| `faiss_index.bin` | ~95 MB | FAISS IndexFlatL2, 47,923개 벡터 |

## 예상 소요 시간

- 데이터 준비: 3-5분 (Drive → 로컬 복사)
- 임베딩 생성: 2-3분 (GPU, 약 300 it/s)
- FAISS 인덱스 생성: 1분 미만

## FastAPI 연동 효과

사전 계산된 임베딩과 인덱스를 로드하면 서버 시작 시간이 3분에서 수 초로 단축되고, 실시간 검색 응답 시간이 1초 이내가 된다.

## 1. 환경 설정 및 경로 초기화

PyTorch, numpy, tqdm 등 필요한 라이브러리를 불러오고 GPU와 경로를 설정한다. Google Drive를 마운트해 모델 체크포인트 경로와 출력 저장 경로를 지정한다. 이미지는 I/O 속도를 위해 로컬에서 읽는다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import time
from google.colab import drive

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Device: {device}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

# Google Drive 마운트
drive.mount('/content/drive')

# 경로 설정
GDRIVE_BASE = '/content/drive/MyDrive/Patron'
LOCAL_BASE = '/content/data'

# 모델은 Google Drive에서
MODEL_PATH = f'{GDRIVE_BASE}/models/best_model.pth'

# 메타데이터와 이미지는 로컬에서 (patron_03 방식)
METADATA_PATH = f'{LOCAL_BASE}/metadata.csv'
IMAGES_PATH = f'{LOCAL_BASE}/images'

# 출력은 Google Drive에
OUTPUT_PATH = f'{GDRIVE_BASE}/data/processed'

print(f"\n✅ 경로 설정 완료")
print(f"   모델: {MODEL_PATH}")
print(f"   메타데이터: {METADATA_PATH}")
print(f"   이미지: {IMAGES_PATH} (로컬!)")
print(f"   출력: {OUTPUT_PATH}")

## 2. 데이터 준비 (patron_03 방식)

Google Drive에서 이미지 tar와 메타데이터를 Colab 로컬로 복사하고 압축 해제한다. 약 49,987개 이미지 파일을 로컬에 두어 임베딩 생성 루프에서 Drive I/O 병목을 제거한다.

In [ ]:
# patron_03 데이터 준비 (로컬 스토리지 사용)
print("📦 patron_03 데이터 복사 중...")
!mkdir -p {LOCAL_BASE}
!cp {GDRIVE_BASE}/images.tar {LOCAL_BASE}/

print("📂 patron_03 압축 해제 중...")
!tar -xf {LOCAL_BASE}/images.tar -C {LOCAL_BASE}/

# 메타데이터 복사
!cp {GDRIVE_BASE}/data/processed/metadata_all.csv {METADATA_PATH}

print("\n✅ 데이터 준비 완료!")
!echo "이미지 개수: $(ls {IMAGES_PATH} | wc -l)"
!echo "메타데이터: $(wc -l < {METADATA_PATH}) lines"

## 3. 모델 정의 (patron_03과 동일)

patron_03과 완전히 동일한 `ChartEmbeddingModel`을 정의한다. 1채널 ResNet18 백본에 L2 정규화 512차원 출력. FastAPI 서버에서도 동일한 클래스 정의를 사용하므로 아키텍처를 변경하면 안 된다.

In [ ]:
class ChartEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=512):
        super(ChartEmbeddingModel, self).__init__()
        # ResNet18 백본 (patron_03과 동일)
        resnet = models.resnet18(pretrained=False)
        # 1채널 입력으로 변경
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # FC 레이어 제거
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.embedding_dim = embedding_dim

    def forward(self, x):
        features = self.features(x)
        embeddings = features.view(features.size(0), -1)
        # L2 정규화 (patron_03과 동일)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings

print("✅ 모델 클래스 정의 완료")

## 4. patron_03 모델 로드

patron_03 Best 체크포인트(Val Loss 0.005174, Epoch 1)를 불러와 모델에 적용하고 eval 모드로 설정한다. 학습된 가중치로 전체 패턴의 512차원 임베딩을 생성할 준비를 마친다.

In [ ]:
print("📥 patron_03 모델 로드 중...")
model = ChartEmbeddingModel(embedding_dim=512).to(device)
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

epoch = checkpoint.get('epoch', '?')
val_loss = checkpoint.get('val_loss', '?')
print(f"✅ 모델 로드 완료 (Epoch {epoch}, Val Loss: {val_loss})")

## 5. 메타데이터 로드

메타데이터 CSV를 불러오고 3개월 수익률이 없는 패턴을 제거한다. FastAPI 서버와 동일한 필터링을 적용해 인덱스 구성이 서빙 환경과 일치하도록 보장한다. 인덱스 i번째 임베딩이 메타데이터 i번째 행과 정확히 대응해야 한다.

In [ ]:
print("\n📊 메타데이터 로드 중...")
metadata = pd.read_csv(METADATA_PATH)
metadata = metadata.dropna(subset=['return_3m'])  # FastAPI와 동일한 필터링
metadata = metadata.reset_index(drop=True)

print(f"✅ 메타데이터 로드 완료: {len(metadata):,}개 패턴")
print(f"   컬럼: {list(metadata.columns)}")
print(f"\n📊 샘플 데이터:")
print(metadata.head())

## 6. 전체 임베딩 생성 (GPU 기준 2-3분)

47,923개 패턴 전체를 모델에 통과시켜 (47,923×512) float32 임베딩 배열을 생성한다. 이미지 로드 실패 시 제로 벡터로 채워 인덱스 크기를 메타데이터 행 수와 동일하게 유지한다. GPU 기준 약 300 it/s, 2-3분 소요.

In [ ]:
print("\n" + "="*60)
print("🔥 전체 패턴 임베딩 생성 시작 (patron_03 방식)")
print("="*60)
print(f"총 패턴 수: {len(metadata):,}개")
print(f"Device: {device}")

embeddings_list = []
failed_indices = []
failed_count = 0
start_time = time.time()

with torch.no_grad():
    for idx in tqdm(range(len(metadata)), desc="임베딩 생성", ncols=100):
        # patron_03 방식: ticker_{pattern_id}.npy (로컬에서 읽기)
        ticker = metadata.loc[idx, 'ticker']
        pattern_id = metadata.loc[idx, 'pattern_id']
        img_path = f"{IMAGES_PATH}/{ticker}_{pattern_id}.npy"

        try:
            # 이미지 로드
            img = np.load(img_path)

            # 텐서 변환 (patron_03 방식과 동일)
            img_tensor = torch.FloatTensor(img).unsqueeze(0).unsqueeze(0).to(device) / 255.0

            # 임베딩 추출
            embedding = model(img_tensor)
            embeddings_list.append(embedding.cpu().numpy()[0])

        except Exception as e:
            # 이미지 로드 실패 시 제로 벡터
            if failed_count < 10:  # 처음 10개만 출력
                print(f"\n⚠️ 패턴 {ticker}_{pattern_id} (idx={idx}) 로드 실패: {e}")
            embeddings_list.append(np.zeros(512, dtype=np.float32))
            failed_indices.append(idx)
            failed_count += 1

# numpy 배열로 변환
embeddings = np.vstack(embeddings_list).astype('float32')
elapsed_time = time.time() - start_time

print(f"\n✅ 임베딩 생성 완료!")
print(f"   소요 시간: {elapsed_time/60:.1f}분")
print(f"   Shape: {embeddings.shape}")
print(f"   성공: {len(embeddings) - failed_count:,}개")
print(f"   실패: {failed_count}개")

if failed_count > 0:
    print(f"\n⚠️ 실패한 인덱스 (처음 20개): {failed_indices[:20]}")

# 메모리 정리
torch.cuda.empty_cache()

## 7. 임베딩 품질 검증

생성된 임베딩의 최솟값, 최댓값, L2 norm을 출력해 품질을 검증한다. L2 정규화가 올바르게 적용됐다면 각 벡터의 norm이 1.0에 가까워야 한다. 제로 벡터 개수로 이미지 로드 실패율도 확인한다.

In [ ]:
# 임베딩 통계
print("\n📊 임베딩 통계:")
print(f"   최소값: {embeddings.min():.4f}")
print(f"   최대값: {embeddings.max():.4f}")
print(f"   평균: {embeddings.mean():.4f}")
print(f"   표준편차: {embeddings.std():.4f}")

# L2 norm 확인 (모두 1.0에 가까워야 함)
norms = np.linalg.norm(embeddings, axis=1)
print(f"\n🔍 L2 Norm 검증:")
print(f"   평균 L2 norm: {norms.mean():.4f} (1.0에 가까워야 함)")
print(f"   최소 L2 norm: {norms.min():.4f}")
print(f"   최대 L2 norm: {norms.max():.4f}")

# 제로 벡터 개수 확인
zero_vectors = np.sum(np.all(embeddings == 0, axis=1))
print(f"\n⚠️ 제로 벡터 개수: {zero_vectors} (실패한 패턴)")

## 8. Google Drive에 저장

임베딩을 `embeddings.npy`로 저장하고 FAISS IndexFlatL2를 생성해 `faiss_index.bin`으로 저장한다. `IndexFlatL2`는 브루트포스 정확 탐색으로 약 50k 벡터 규모에서 충분히 빠르다. 두 파일을 FastAPI 데이터 폴더에 복사하면 배포 완료다.

In [ ]:
# 1. embeddings.npy 저장
print("\n" + "="*60)
print("💾 임베딩 저장 중...")
print("="*60)

embeddings_save_path = f"{OUTPUT_PATH}/embeddings.npy"
np.save(embeddings_save_path, embeddings)
print(f"✅ embeddings.npy 저장 완료")
print(f"   경로: {embeddings_save_path}")

# 파일 크기 확인
file_size_mb = os.path.getsize(embeddings_save_path) / (1024 * 1024)
print(f"   크기: {file_size_mb:.1f} MB")
print(f"   Shape: {embeddings.shape}")

# 2. FAISS 인덱스 생성 및 저장
# faiss 설치 (Colab에서 필요)
!pip install -q faiss-cpu

print("\n" + "="*60)
print("🔍 FAISS 인덱스 생성 중...")
print("="*60)

import faiss

# L2 인덱스 생성
dimension = 512
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# 저장
faiss_save_path = f"{OUTPUT_PATH}/faiss_index.bin"
faiss.write_index(index, faiss_save_path)

print(f"✅ FAISS 인덱스 저장 완료")
print(f"   경로: {faiss_save_path}")
print(f"   벡터 수: {index.ntotal:,}개")

# 파일 크기 확인
faiss_size_mb = os.path.getsize(faiss_save_path) / (1024 * 1024)
print(f"   크기: {faiss_size_mb:.1f} MB")

print("\n" + "="*60)
print("🎉 모든 파일 저장 완료!")
print("="*60)

## 9. 완료 및 FastAPI 배포 안내

생성된 파일 목록과 크기를 출력하고 FastAPI 배포를 위한 다음 단계를 안내한다. `embeddings.npy`와 `faiss_index.bin`을 FastAPI 데이터 폴더에 복사하고 서버 시작 코드에서 임베딩 재생성 대신 파일 로드로 변경하면 응답 시간이 3분에서 1초 이내로 단축된다.

In [ ]:
print("\n" + "="*60)
print("✅ patron_13 임베딩 재생성 완료!")
print("="*60)

print(f"\n📂 생성된 파일:")
print(f"   1. embeddings.npy")
print(f"      - 경로: {embeddings_save_path}")
print(f"      - 크기: {file_size_mb:.1f} MB")
print(f"      - Shape: {embeddings.shape}")
print(f"")
print(f"   2. faiss_index.bin")
print(f"      - 경로: {faiss_save_path}")
print(f"      - 크기: {faiss_size_mb:.1f} MB")
print(f"      - 벡터 수: {index.ntotal:,}개")

print(f"\n📥 다음 단계:")
print(f"   1. Google Drive에서 두 파일 다운로드")
print(f"      - embeddings.npy")
print(f"      - faiss_index.bin")
print(f"")
print(f"   2. FastAPI 폴더에 복사:")
print(f"      ~/Desktop/Dipping/Dipping-AI/patron/patron_fastapi/data/")
print(f"")
print(f"   3. FastAPI main.py 수정:")
print(f"      - 매번 임베딩 생성 → embeddings.npy 로드로 변경")
print(f"      - 3분 → 1초로 단축!")
print(f"")
print(f"   4. FastAPI 서버 재시작 및 테스트")

print("\n" + "="*60)
print(f"🎉 patron_13 완료! (patron_03 모델 기반)")
print("="*60)